## Silver Layer — Cleansed & Typed Data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Load Bronze
bronze_df = spark.table(
    "healthflow_catalog.bronze.patients_raw"
)

print(f"Bronze records: {bronze_df.count():,}")
print(f"Columns: {bronze_df.columns}")

# Silver transformations
silver_df = bronze_df \
    .withColumn("name",
        F.initcap(F.col("Name"))) \
    .withColumn("age",
        F.col("Age").cast(IntegerType())) \
    .withColumn("gender",
        F.lower(F.col("Gender"))) \
    .withColumn("doctor",
        F.initcap(F.col("Doctor"))) \
    .withColumn("hospital",
        F.col("Hospital")) \
    .withColumn("medication",
        F.col("Medication")) \
    .withColumn("length_of_stay",
        F.datediff(
            F.col("discharge_date"),
            F.col("date_of_admission")
        )) \
    .withColumn("billing_amount",
        F.round(F.col("billing_amount"), 2)) \
    .withColumn("room_number",
        F.col("room_number").cast(IntegerType())) \
    .select(
        "name", "age", "gender",
        "blood_type", "medical_condition",
        "date_of_admission", "discharge_date",
        "length_of_stay", "doctor",
        "hospital", "insurance_provider",
        "billing_amount", "room_number",
        "admission_type", "medication",
        "test_results", "ingestion_timestamp"
    ) \
    .dropDuplicates() \
    .dropna()

print(f"Silver records: {silver_df.count():,}")

# Save
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "healthflow_catalog.silver.patients_clean"
    )

print("✅ Silver layer complete!")

Bronze records: 55,500
Columns: ['Name', 'Age', 'Gender', 'blood_type', 'medical_condition', 'date_of_admission', 'Doctor', 'Hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'Medication', 'test_results', 'ingestion_timestamp', 'source_file']
Silver records: 54,966
✅ Silver layer complete!
